In [1]:
import os
import gc
from glob import glob

import numpy as np
import pandas as pd

from datasets import Dataset, DatasetDict, load_metric

from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, 
                          Seq2SeqTrainer, DataCollatorForSeq2Seq)

import warnings
warnings.filterwarnings('ignore')

In [4]:
config = {
    "output_dir": "t5_small_lab3_finetune",
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    'num_train_epochs': 4,
    "train_batch_size": 1,
    "eval_batch_size": 1,
    "max_seq_length": 1024,
    "overwrite_output_dir": True,
    "reprocess_input_data": True,
    "fp16": True
}

In [2]:
prefix_val = "summarize"

df = pd.read_parquet("arxiv_title_generation.parquet")

print("\nDataframe memory usage")
print(df.memory_usage(deep = True))

print(f"Dataframe shape: {df.shape}\n")

print(df.head())


Dataframe memory usage
Index               132
target_text     1963896
input_text     17433653
prefix           954492
dtype: int64
Dataframe shape: (14462, 3)

                                         target_text  \
0  Improved Mispronunciation detection system usi...   
1  Improving Factored Hybrid HMM Acoustic Modelin...   
2  Disentangling Style and Speaker Attributes for...   
3  Synthetic speech detection using meta-learning...   
4  A Pre-trained Audio-Visual Transformer for Emo...   

                                          input_text     prefix  
0  This report proposes state-of-the-art research...  summarize  
1  In this work, we show that a factored hybrid h...  summarize  
2  End-to-end neural TTS has shown improved perfo...  summarize  
3  Recent works on speech spoofing countermeasure...  summarize  
4  In this paper, we introduce a pretrained audio...  summarize  


In [3]:
test_size = 0.2
test_df = df.sample(frac = test_size, random_state = 1970)
train_df = df.drop(index = test_df.index)

print(f"Training instance count: {len(train_df)}\nTest instance count: {len(test_df)}\n")

train_dataset = Dataset.from_dict(train_df)
test_dataset = Dataset.from_dict(test_df)
arxiv_title_dict = DatasetDict({"train": train_dataset,"test": test_dataset})

print(arxiv_title_dict)

Training instance count: 11570
Test instance count: 2892

DatasetDict({
    train: Dataset({
        features: ['target_text', 'input_text', 'prefix'],
        num_rows: 11570
    })
    test: Dataset({
        features: ['target_text', 'input_text', 'prefix'],
        num_rows: 2892
    })
})


In [5]:
tokenizer_name = "google/t5-efficient-mini"
#tokenizer_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast = False)

def preprocess_function(examples):
    inputs = [f"{prefix_val}: " + doc for doc in examples["input_text"]]
    model_inputs = tokenizer(inputs,
                             max_length = config['max_seq_length'],
                             padding = True,
                             truncation = True)

    labels = tokenizer(text_target = examples["target_text"],
                       max_length = config["max_seq_length"] // 8,
                       padding = True,
                       truncation = True)
    
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_arxiv = arxiv_title_dict.map(preprocess_function, batched = True)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Map:   0%|          | 0/11570 [00:00<?, ? examples/s]

Map:   0%|          | 0/2892 [00:00<?, ? examples/s]

In [6]:
rouge = load_metric("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, 
                                           skip_special_tokens = True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(labels, 
                                            skip_special_tokens = True)

    # Compute ROUGE-1 scores
    rouge_scores = rouge.compute(predictions = decoded_preds, 
                                 references = decoded_labels, 
                                 rouge_types = ["rouge1"])["rouge1"]

    # Calculate the mean ROUGE-1 F1 score
    rouge1_f1 = np.mean([score["f"] for score in rouge_scores])

    # Rounds the result to 4 decimal places for cleaner output, and returns it.
    return {"rouge1_f1": round(rouge1_f1, 4)}

In [8]:
model_type = "google/t5-efficient-mini"
model = AutoModelForSeq2SeqLM.from_pretrained("./t5_small_lab3_finetune/checkpoint-11500")
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer, model = model_type)

In [17]:
idx = 100
text = "summarize: " + tokenized_arxiv['test'][idx]['input_text']
actual_text = tokenized_arxiv['test'][idx]['target_text']
print(text)

summarize: To estimate the direction of arrival (DOA) of multiple speakers with methods that use prototype transfer functions, frequency-dependent spatial spectra (SPS) are usually constructed. To make the DOA estimation robust, SPS from different frequencies can be combined. According to how the SPS are combined, frequency fusion mechanisms are categorized into narrowband, broadband, or speaker-grouped, where the latter mechanism requires a speaker-wise grouping of frequencies. For a binaural hearing aid setup, in this paper we propose an interaural time difference (ITD)-based speaker-grouped frequency fusion mechanism. By exploiting the DOA dependence of ITDs, frequencies can be grouped according to a common ITD and be used for DOA estimation of the respective speaker. We apply the proposed ITD-based speaker-grouped frequency fusion mechanism for different DOA estimation methods, namely the multiple signal classification, steered response power and a recently published method based o

In [18]:
from transformers import pipeline

summarizer = pipeline("summarization", model = "./t5_small_lab3_finetune/checkpoint-11500")
pred = summarizer(text)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [19]:
print(f"Actual Title: {actual_text}")
print(f"Predicted Title: {pred[0]['summary_text']}")

Actual Title: Comparison of Frequency-Fusion Mechanisms for Binaural Direction-of-Arrival Estimation for Multiple Speakers
Predicted Title: Integrated Audio-grouped frequency fusion mechanism for binaural hearing aid setup
